# Benchmark analysis

Prototype notebook for Issue #10. It loads the benchmark runner CSV from Issue #9 and produces performance profiles, data profiles, comparison plots, and paired statistical tests for DSDA vs RBFOpt.

This notebook is deliberately organized as plain Python functions first and example calls second. When we are ready to make this production code, the function cells can move into `benchmark_analysis.py` and the notebook can become a thin report layer.

## Inputs

Run `benchmark.py` first, then either set `RESULTS_CSV` to the exact output path or leave it as `None` to use the newest `results/benchmark_*.csv` file.

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats


RESULTS_CSV = None
RESULTS_DIR = Path("results")
GRIDS_DIR = Path("benchmark_grids")
FIGURE_DIR = Path("results/figures")

METHOD_ORDER = ("DSDA", "RBFOpt")
PRIMARY_TOLERANCE = 1e-5
TOLERANCES = (1e-1, 1e-3, 1e-5, 1e-7)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "legend.fontsize": 10,
        "font.size": 10,
    }
)

## Load and normalize benchmark results

In [ ]:
REQUIRED_COLUMNS = {
    "csv_name",
    "method",
    "seed",
    "n_starts",
    "max_evals",
    "best_value",
    "best_point",
    "total_evals",
    "wall_time_s",
    "found_optimum",
    "incumbent_trajectory",
}


def find_latest_results_csv(results_csv=None, results_dir=RESULTS_DIR):
    """Return an explicit CSV path or the newest benchmark CSV in results_dir."""
    if results_csv is not None:
        path = Path(results_csv)
        if not path.exists():
            raise FileNotFoundError(f"Benchmark results CSV does not exist: {path}")
        return path

    candidates = sorted(Path(results_dir).glob("benchmark_*.csv"))
    if not candidates:
        raise FileNotFoundError(
            "No benchmark CSV found. Run benchmark.py first or set RESULTS_CSV "
            "to an explicit results file."
        )
    return max(candidates, key=lambda path: path.stat().st_mtime)


def _json_or_value(value):
    if pd.isna(value):
        return value
    if isinstance(value, (list, tuple, dict)):
        return value
    try:
        return json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return value


def load_benchmark_results(results_csv):
    """Load the unified benchmark CSV written by benchmark.py."""
    df = pd.read_csv(results_csv)
    missing = REQUIRED_COLUMNS.difference(df.columns)
    if missing:
        missing_text = ", ".join(sorted(missing))
        raise ValueError(f"Results CSV is missing required columns: {missing_text}")

    df = df.copy()
    df["method"] = df["method"].astype(str)
    for column in ("seed", "n_starts", "max_evals", "total_evals"):
        df[column] = pd.to_numeric(df[column], errors="raise").astype(int)
    for column in ("best_value", "wall_time_s"):
        df[column] = pd.to_numeric(df[column], errors="raise")

    df["found_optimum"] = (
        df["found_optimum"].astype(str).str.lower().isin({"true", "1", "yes"})
    )
    df["best_point"] = df["best_point"].apply(_json_or_value)
    df["incumbent_trajectory"] = df["incumbent_trajectory"].apply(_json_or_value)
    return df

In [ ]:
def load_grid_metadata(grids_dir=GRIDS_DIR):
    """Load known optima and dimensions from benchmark grid CSV files."""
    rows = []
    for path in sorted(Path(grids_dir).glob("*.csv")):
        grid_df = pd.read_csv(path)
        value_column = "z_norm" if "z_norm" in grid_df.columns else grid_df.columns[-1]
        coordinate_columns = [column for column in grid_df.columns if column != value_column]
        rows.append(
            {
                "csv_name": path.name,
                "target_value": float(grid_df[value_column].min()),
                "dimension": len(coordinate_columns),
                "grid_points": len(grid_df),
            }
        )

    if not rows:
        raise FileNotFoundError(f"No grid CSV files found in {grids_dir}")
    return pd.DataFrame(rows)


def attach_grid_metadata(results_df, grids_dir=GRIDS_DIR):
    """Attach known target values and dimensions to benchmark results."""
    metadata = load_grid_metadata(grids_dir)
    merged = results_df.merge(metadata, on="csv_name", how="left")

    missing_target = merged["target_value"].isna()
    if missing_target.any():
        observed_best = merged.groupby("csv_name")["best_value"].transform("min")
        merged.loc[missing_target, "target_value"] = observed_best[missing_target]
        merged.loc[missing_target, "dimension"] = 1
        print(
            "Warning: some grids were not found in GRIDS_DIR. "
            "Observed best values were used as fallback targets."
        )

    merged["dimension"] = merged["dimension"].astype(int)
    return merged

## Solve status helpers

For each run, `evals_to_solve` is the first incumbent-trajectory evaluation that reaches the known grid optimum within the requested tolerance. The benchmark runner is minimizing `z_norm`, so a value is solved when it is no larger than `target_value + tolerance * max(1, abs(target_value))`.

In [ ]:
def tolerance_window(target_value, tolerance):
    """Return the absolute solve window for a target and relative tolerance."""
    return tolerance * max(1.0, abs(float(target_value)))


def first_eval_within_tolerance(trajectory, target_value, tolerance):
    """Return the 1-based first evaluation that reaches the tolerance."""
    if not isinstance(trajectory, (list, tuple)):
        return math.nan

    threshold = float(target_value) + tolerance_window(target_value, tolerance)
    for eval_number, value in enumerate(trajectory, start=1):
        if float(value) <= threshold:
            return eval_number
    return math.nan


def annotate_solve_status(results_df, tolerance=PRIMARY_TOLERANCE):
    """Add solved and evals_to_solve columns for one tolerance level."""
    df = results_df.copy()
    df["tolerance"] = tolerance
    df["evals_to_solve"] = [
        first_eval_within_tolerance(trajectory, target, tolerance)
        for trajectory, target in zip(df["incumbent_trajectory"], df["target_value"])
    ]

    exact_fallback = df["evals_to_solve"].isna() & df["found_optimum"]
    df.loc[exact_fallback, "evals_to_solve"] = df.loc[exact_fallback, "total_evals"]
    df["solved"] = df["evals_to_solve"].notna()
    return df

## Performance profiles

For a fixed budget, each `(grid, seed)` pair is a problem instance. The performance ratio is the method's evaluations-to-solve divided by the best evaluations-to-solve achieved by any method on that instance. Unsolved runs count as not solved at every finite ratio.

In [ ]:
def performance_ratio_table(results_df, tolerance=PRIMARY_TOLERANCE, budget=None):
    """Build a method-by-problem table of Dolan-More performance ratios."""
    data = annotate_solve_status(results_df, tolerance)
    if budget is not None:
        data = data[data["max_evals"] == budget]

    pivot = data.pivot_table(
        index=["csv_name", "seed"],
        columns="method",
        values="evals_to_solve",
        aggfunc="min",
    )
    best = pivot.min(axis=1, skipna=True)
    pivot = pivot.loc[best.notna()]
    if pivot.empty:
        return pivot
    return pivot.div(best.loc[pivot.index], axis=0)


def _finite_profile_grid(ratios):
    finite = ratios.to_numpy(dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return np.array([1.0, 2.0])
    return np.linspace(1.0, max(2.0, float(finite.max())), 200)


def plot_performance_profiles(
    results_df,
    tolerance=PRIMARY_TOLERANCE,
    budgets=None,
    method_order=METHOD_ORDER,
):
    """Plot performance profiles by budget level."""
    if budgets is None:
        budgets = sorted(results_df["max_evals"].unique())

    fig, axes = plt.subplots(
        1,
        len(budgets),
        figsize=(4.2 * len(budgets), 3.4),
        sharey=True,
        squeeze=False,
    )

    for ax, budget in zip(axes.ravel(), budgets):
        ratios = performance_ratio_table(results_df, tolerance, budget)
        tau_values = _finite_profile_grid(ratios)
        methods = [method for method in method_order if method in ratios.columns]

        if ratios.empty or not methods:
            ax.text(0.5, 0.5, "No solved instances", ha="center", va="center")
        else:
            denominator = len(ratios)
            for method in methods:
                method_ratios = ratios[method].to_numpy(dtype=float)
                fractions = [np.sum(method_ratios <= tau) / denominator for tau in tau_values]
                ax.step(tau_values, fractions, where="post", label=method, linewidth=2)

        ax.set_title(f"max_evals = {budget}")
        ax.set_xlabel("performance ratio tau")
        ax.set_ylim(0.0, 1.02)
        ax.legend(loc="lower right")

    axes.ravel()[0].set_ylabel("fraction solved")
    fig.suptitle(f"Performance profiles, tolerance={tolerance:g}")
    fig.tight_layout()
    return fig

## Data profiles

Data profiles use the longest available budget per `(grid, seed, method)` and plot the fraction solved against evaluations normalized by problem dimension.

In [ ]:
def longest_budget_rows(results_df):
    """Keep the row with the largest budget for each grid, seed, and method."""
    idx = results_df.groupby(["csv_name", "seed", "method"])["max_evals"].idxmax()
    return results_df.loc[idx].copy()


def data_profile_frame(results_df, tolerance=PRIMARY_TOLERANCE):
    """Return solve times normalized by dimension for data profiles."""
    data = annotate_solve_status(longest_budget_rows(results_df), tolerance)
    data["normalized_evals_to_solve"] = data["evals_to_solve"] / data["dimension"]
    return data


def plot_data_profiles(
    results_df,
    tolerances=TOLERANCES,
    method_order=METHOD_ORDER,
):
    """Plot More-Wild data profiles for one or more tolerance levels."""
    fig, axes = plt.subplots(
        1,
        len(tolerances),
        figsize=(4.2 * len(tolerances), 3.4),
        sharey=True,
        squeeze=False,
    )

    for ax, tolerance in zip(axes.ravel(), tolerances):
        data = data_profile_frame(results_df, tolerance)
        methods = [method for method in method_order if method in set(data["method"])]
        finite = data["normalized_evals_to_solve"].dropna()
        max_x = max(1.0, float(finite.max())) if not finite.empty else 1.0
        x_values = np.linspace(0.0, max_x, 200)

        for method in methods:
            method_values = data.loc[
                data["method"] == method,
                "normalized_evals_to_solve",
            ].to_numpy(dtype=float)
            denominator = len(method_values)
            fractions = [np.sum(method_values <= x_value) / denominator for x_value in x_values]
            ax.step(x_values, fractions, where="post", label=method, linewidth=2)

        ax.set_title(f"tolerance = {tolerance:g}")
        ax.set_xlabel("evaluations / dimension")
        ax.set_ylim(0.0, 1.02)
        ax.legend(loc="lower right")

    axes.ravel()[0].set_ylabel("fraction solved")
    fig.suptitle("Data profiles")
    fig.tight_layout()
    return fig

## Additional comparison plots

In [ ]:
def plot_best_value_boxplots(results_df, method_order=METHOD_ORDER):
    """Box plots of final best_value by method at each budget."""
    budgets = sorted(results_df["max_evals"].unique())
    fig, axes = plt.subplots(
        1,
        len(budgets),
        figsize=(4.0 * len(budgets), 3.4),
        sharey=True,
        squeeze=False,
    )

    for ax, budget in zip(axes.ravel(), budgets):
        data = results_df[results_df["max_evals"] == budget]
        methods = [method for method in method_order if method in set(data["method"])]
        values = [data.loc[data["method"] == method, "best_value"] for method in methods]
        ax.boxplot(values, labels=methods, showmeans=True)
        ax.set_title(f"max_evals = {budget}")
        ax.set_xlabel("method")

    axes.ravel()[0].set_ylabel("best_value")
    fig.suptitle("Best value distributions")
    fig.tight_layout()
    return fig


def plot_winner_heatmap(results_df, method_a="DSDA", method_b="RBFOpt"):
    """Heatmap of average winner by grid and budget for two methods."""
    pivot = results_df.pivot_table(
        index=["csv_name", "max_evals"],
        columns="method",
        values="best_value",
        aggfunc="mean",
    )
    if method_a not in pivot.columns or method_b not in pivot.columns:
        raise ValueError(f"Expected both {method_a} and {method_b} in results")

    # Positive deltas favor method_a because the benchmark is minimizing.
    delta = (pivot[method_b] - pivot[method_a]).unstack("max_evals")
    fig_height = max(4.0, 0.35 * len(delta.index))
    fig, ax = plt.subplots(figsize=(8.0, fig_height))
    image = ax.imshow(delta.to_numpy(), cmap="coolwarm", aspect="auto")
    ax.set_xticks(np.arange(len(delta.columns)), labels=delta.columns)
    ax.set_yticks(np.arange(len(delta.index)), labels=delta.index)
    ax.set_xlabel("max_evals")
    ax.set_ylabel("grid")
    ax.set_title(f"Average winner: positive favors {method_a}")

    for y_index, grid_name in enumerate(delta.index):
        for x_index, budget in enumerate(delta.columns):
            value = delta.loc[grid_name, budget]
            if pd.isna(value):
                label = "missing"
            elif np.isclose(value, 0.0):
                label = "tie"
            else:
                label = method_a if value > 0 else method_b
            ax.text(x_index, y_index, label, ha="center", va="center", fontsize=8)

    fig.colorbar(image, ax=ax, label=f"mean {method_b} best_value - mean {method_a} best_value")
    fig.tight_layout()
    return fig


def plot_wall_time_scatter(results_df, method_a="DSDA", method_b="RBFOpt"):
    """Scatter plot of paired wall time per grid, seed, and budget."""
    paired = results_df.pivot_table(
        index=["csv_name", "seed", "max_evals"],
        columns="method",
        values="wall_time_s",
        aggfunc="mean",
    ).dropna(subset=[method_a, method_b])

    fig, ax = plt.subplots(figsize=(5.0, 4.2))
    ax.scatter(paired[method_a], paired[method_b], alpha=0.7)
    max_time = float(np.nanmax(paired[[method_a, method_b]].to_numpy()))
    ax.plot([0, max_time], [0, max_time], color="black", linestyle="--", linewidth=1)
    ax.set_xlabel(f"{method_a} wall_time_s")
    ax.set_ylabel(f"{method_b} wall_time_s")
    ax.set_title("Paired wall-time comparison")
    fig.tight_layout()
    return fig

In [ ]:
def trajectory_mean_frame(results_df, grid_name=None, budget=None):
    """Explode incumbent trajectories and average them over seeds."""
    data = results_df.copy()
    if grid_name is None:
        grid_name = sorted(data["csv_name"].unique())[0]
    if budget is None:
        budget = int(data["max_evals"].max())

    data = data[(data["csv_name"] == grid_name) & (data["max_evals"] == budget)]
    rows = []
    for row in data.itertuples(index=False):
        trajectory = getattr(row, "incumbent_trajectory")
        if not isinstance(trajectory, (list, tuple)):
            continue
        for eval_number, incumbent in enumerate(trajectory, start=1):
            rows.append(
                {
                    "csv_name": row.csv_name,
                    "method": row.method,
                    "seed": row.seed,
                    "max_evals": row.max_evals,
                    "eval_number": eval_number,
                    "incumbent": float(incumbent),
                }
            )

    if not rows:
        raise ValueError(f"No trajectories found for grid={grid_name}, budget={budget}")

    trajectory_df = pd.DataFrame(rows)
    return (
        trajectory_df.groupby(["method", "eval_number"], as_index=False)["incumbent"]
        .mean()
        .sort_values(["method", "eval_number"])
    )


def plot_convergence_curves(results_df, grid_name=None, budget=None, method_order=METHOD_ORDER):
    """Plot mean incumbent value vs evaluation count for one grid and budget."""
    if grid_name is None:
        grid_name = sorted(results_df["csv_name"].unique())[0]
    if budget is None:
        budget = int(results_df["max_evals"].max())

    means = trajectory_mean_frame(results_df, grid_name=grid_name, budget=budget)
    fig, ax = plt.subplots(figsize=(5.5, 4.0))
    for method in method_order:
        method_data = means[means["method"] == method]
        if method_data.empty:
            continue
        ax.plot(
            method_data["eval_number"],
            method_data["incumbent"],
            marker="o",
            linewidth=2,
            label=method,
        )

    ax.set_xlabel("evaluation count")
    ax.set_ylabel("mean incumbent best_value")
    ax.set_title(f"Convergence curves: {grid_name}, max_evals={budget}")
    ax.legend()
    fig.tight_layout()
    return fig

## Statistical tests

The primary comparison is Wilcoxon signed-rank by budget, pairing DSDA and RBFOpt on each grid after averaging over seeds. The rank-biserial effect size is positive when `method_a` has lower `best_value` than `method_b`.

In [ ]:
def holm_bonferroni(results_df, p_col="p_value", alpha=0.05):
    """Add Holm-Bonferroni adjusted p-values and reject flags."""
    if results_df.empty:
        return results_df.copy()

    adjusted = results_df.copy()
    order = adjusted[p_col].sort_values().index.tolist()
    m = len(order)
    running_max = 0.0
    adjusted["holm_p_value"] = np.nan

    for rank, index in enumerate(order, start=1):
        raw_adjusted = min(1.0, (m - rank + 1) * adjusted.loc[index, p_col])
        running_max = max(running_max, raw_adjusted)
        adjusted.loc[index, "holm_p_value"] = running_max

    adjusted["holm_reject_0.05"] = adjusted["holm_p_value"] <= alpha
    return adjusted.sort_values("max_evals").reset_index(drop=True)


def rank_biserial_from_differences(differences):
    """Rank-biserial correlation from paired differences."""
    differences = np.asarray(differences, dtype=float)
    differences = differences[~np.isclose(differences, 0.0)]
    if differences.size == 0:
        return 0.0

    ranks = stats.rankdata(np.abs(differences))
    positive = ranks[differences > 0].sum()
    negative = ranks[differences < 0].sum()
    return float((positive - negative) / ranks.sum())


def wilcoxon_by_budget(results_df, method_a="DSDA", method_b="RBFOpt"):
    """Run paired Wilcoxon signed-rank tests at each budget level."""
    aggregated = (
        results_df.groupby(["csv_name", "max_evals", "method"], as_index=False)["best_value"]
        .mean()
    )
    rows = []
    for budget, budget_df in aggregated.groupby("max_evals"):
        paired = budget_df.pivot(index="csv_name", columns="method", values="best_value")
        paired = paired.dropna(subset=[method_a, method_b])
        if paired.empty:
            continue

        differences = paired[method_b] - paired[method_a]
        if np.allclose(differences, 0.0):
            statistic = 0.0
            p_value = 1.0
        else:
            result = stats.wilcoxon(
                paired[method_a],
                paired[method_b],
                zero_method="wilcox",
                alternative="two-sided",
            )
            statistic = float(result.statistic)
            p_value = float(result.pvalue)

        rows.append(
            {
                "max_evals": int(budget),
                "n_pairs": len(paired),
                "method_a": method_a,
                "method_b": method_b,
                "mean_difference_b_minus_a": float(differences.mean()),
                "wilcoxon_statistic": statistic,
                "p_value": p_value,
                "rank_biserial_effect": rank_biserial_from_differences(differences),
            }
        )

    return holm_bonferroni(pd.DataFrame(rows))


def friedman_by_budget(results_df):
    """Run Friedman tests by budget when three or more methods are present."""
    methods = sorted(results_df["method"].unique())
    if len(methods) < 3:
        raise ValueError("Friedman tests require at least three methods.")

    aggregated = (
        results_df.groupby(["csv_name", "max_evals", "method"], as_index=False)["best_value"]
        .mean()
    )
    rows = []
    for budget, budget_df in aggregated.groupby("max_evals"):
        paired = budget_df.pivot(index="csv_name", columns="method", values="best_value")
        paired = paired.dropna(subset=methods)
        if paired.empty:
            continue
        result = stats.friedmanchisquare(*(paired[method].to_numpy() for method in methods))
        rows.append(
            {
                "max_evals": int(budget),
                "n_blocks": len(paired),
                "methods": methods,
                "friedman_statistic": float(result.statistic),
                "p_value": float(result.pvalue),
            }
        )
    return pd.DataFrame(rows)

## Run the analysis

In [ ]:
results_path = find_latest_results_csv(RESULTS_CSV)
raw_results = load_benchmark_results(results_path)
results = attach_grid_metadata(raw_results, GRIDS_DIR)

print(f"Loaded {len(results):,} benchmark rows from {results_path}")
print(f"Grids: {results['csv_name'].nunique()}, methods: {sorted(results['method'].unique())}")
results.head()

In [ ]:
solved_summary = (
    annotate_solve_status(results, PRIMARY_TOLERANCE)
    .groupby(["method", "max_evals"])["solved"]
    .mean()
    .unstack("max_evals")
)
solved_summary

In [ ]:
plot_performance_profiles(results, tolerance=PRIMARY_TOLERANCE);

In [ ]:
plot_data_profiles(results, tolerances=TOLERANCES);

In [ ]:
plot_best_value_boxplots(results);

In [ ]:
plot_winner_heatmap(results);

In [ ]:
plot_wall_time_scatter(results);

In [ ]:
plot_convergence_curves(results);

In [ ]:
wilcoxon_results = wilcoxon_by_budget(results)
wilcoxon_results

## Optional: save figures

Uncomment this cell when the plots look right. The figures are saved at 300 DPI under `results/figures/`.

In [ ]:
# FIGURE_DIR.mkdir(parents=True, exist_ok=True)
# plot_performance_profiles(results, tolerance=PRIMARY_TOLERANCE).savefig(
#     FIGURE_DIR / "performance_profiles.png",
#     bbox_inches="tight",
# )
# plot_data_profiles(results, tolerances=TOLERANCES).savefig(
#     FIGURE_DIR / "data_profiles.png",
#     bbox_inches="tight",
# )
# plot_winner_heatmap(results).savefig(
#     FIGURE_DIR / "winner_heatmap.png",
#     bbox_inches="tight",
# )